# GridSearchCV: Ridge, Lasso e ElasticNet

Vamos comparar três formas de regularização usando os mesmos dados, divisões e métricas:

- **Ridge (L2):** reduz os coeficientes, geralmente sem zerá-los.
- **Lasso (L1):** pode zerar coeficientes e selecionar variáveis.
- **ElasticNet (L1 + L2):** combina as duas penalizações; `l1_ratio` controla essa mistura.

Cada modelo terá sua própria busca, melhores parâmetros, tabela de candidatos e gráfico de coeficientes. Execute as células em ordem. A comparação final usa validação cruzada aninhada para avaliar também a escolha dos hiperparâmetros.

**Objetivo deste notebook:** Comparar Ridge, Lasso e ElasticNet com as mesmas métricas e validação cruzada aninhada, incluindo modelos de referência.


## Abrir cada modelo em sua própria aba

- [Ridge](02a_GridSearchCV_Ridge.ipynb) — abrir em uma aba própria.
- [Lasso](02b_GridSearchCV_Lasso.ipynb) — abrir em uma aba própria.
- [ElasticNet](02c_GridSearchCV_ElasticNet.ipynb) — abrir em uma aba própria.

Este notebook mantém a explicação completa e a comparação final. Cada arquivo acima funciona sozinho, com preparação dos dados, busca e comentários.


## 1. Importações e dados

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.model_selection import KFold, cross_validate
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    PowerTransformer,
    StandardScaler,
    QuantileTransformer,
)

# Localiza este módulo ao executar da raiz do repositório ou da pasta notebooks.
from pathlib import Path
PASTA_ATUAL = Path.cwd().resolve()
PASTA_PROJETO = next(
    pasta
    for base in (PASTA_ATUAL, *PASTA_ATUAL.parents)
    for pasta in (base, base / "03_regularizacao")
    if pasta.name == "03_regularizacao" and (pasta / "src" / "config.py").is_file()
)
if str(PASTA_PROJETO) not in sys.path:
    sys.path.insert(0, str(PASTA_PROJETO))

from src.utils import dataframe_coeficientes
from src.config import DADOS_DIABETES_CATEGORIZADOS
from src.graficos import plot_comparar_metricas_modelos, plot_coeficientes
from src.modelos import (
    RANDOM_STATE,
    grid_search_cv_regressor,
    organiza_resultados,
    construir_pipeline_modelo_regressao,
)

sns.set_theme(palette="bright")

In [ ]:
df = pd.read_parquet(DADOS_DIABETES_CATEGORIZADOS)

df.head()

In [ ]:
df.info()

In [ ]:
X = df.drop(columns="target")
y = df["target"]

## 2. Pré-processamento compartilhado

Os três modelos testam os mesmos três pré-processadores e a mesma opção de transformar o alvo. As transformações ficam dentro do pipeline: são ajustadas apenas nos dados de treino de cada divisão, evitando vazamento de dados.

In [ ]:
colunas_power_transform = ["imc", "ldl", "hdl", "colesterol_total"]

coluna_target = ["target"]

coluna_ordinal_encoder = ["colesterol_hdl_cat"]

coluna_one_hot_encoder = ["sexo"]

colunas_standard_scaler = [
    coluna
    for coluna in df.columns
    if coluna not in colunas_power_transform + coluna_target + coluna_ordinal_encoder + coluna_one_hot_encoder
]

colunas_standard_scaler

In [ ]:
categorias_ordinal_encoder = [
    ["2-3", "4-5", "6+"],
]

In [ ]:
preprocessamento_categoricas = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ],
    remainder="passthrough",
)

preprocessamento_simples = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
        ("standard_scaler", StandardScaler(), X.columns.difference(coluna_ordinal_encoder + coluna_one_hot_encoder)),
    ],
    remainder="passthrough",
)

preprocessamento_completo = ColumnTransformer(
    [
        ("power_transform", PowerTransformer(method="box-cox"), colunas_power_transform),
        ("standard_scaler", StandardScaler(), colunas_standard_scaler),
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ]
)

target_transformer = QuantileTransformer(n_quantiles=20, output_distribution="normal", random_state=RANDOM_STATE)

## 3. Regras comuns das buscas

`alpha` controla a intensidade da regularização. Todos testam os mesmos valores; ElasticNet testa também `l1_ratio`.

A função `grid_search_cv_regressor` usa cinco divisões com embaralhamento e semente fixa, calcula R², MAE e RMSE e escolhe pelo **menor RMSE**. No scikit-learn os erros são armazenados com sinal negativo: `best_score_` é o RMSE negativo. Nas tabelas abaixo, MAE e RMSE aparecem positivos.

As buscas nesta etapa usam todos os dados para explorar configurações e coeficientes. Seus melhores escores não são uma avaliação independente; essa avaliação vem na seção final.

In [ ]:
ALPHAS = [0.01, 0.05, 0.1, 1.0, 5.0, 10.0]
PREPROCESSADORES = [
    preprocessamento_categoricas,
    preprocessamento_simples,
    preprocessamento_completo,
]
TRANSFORMADORES_ALVO = [target_transformer, None]

# As mesmas divisões externas serão usadas por todos na comparação final.
cv_externa = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
METRICAS = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]

## 4. Ridge

A penalização L2 reduz a magnitude dos coeficientes.

Seguimos os mesmos passos: definir a grade, treinar a busca, examinar os candidatos e visualizar os coeficientes.

In [ ]:
# 1. Grade exclusiva do Ridge; as opções comuns são iguais nos três modelos.
param_grid_ridge = {
    "regressor__reg__alpha": ALPHAS,
    "regressor__preprocessor": PREPROCESSADORES,
    "transformer": TRANSFORMADORES_ALVO,
}

# Mantemos o wrapper do alvo mesmo quando a grade escolhe transformer=None.
grid_search_ridge = grid_search_cv_regressor(
    regressor=Ridge(),
    param_grid=param_grid_ridge,
    preprocessor=preprocessamento_categoricas,
    target_transformer=target_transformer,
)

In [ ]:
# 2. Ajusta cada configuração por validação cruzada e reajusta a melhor em X, y.
grid_search_ridge.fit(X, y)

In [ ]:
# 3. Parâmetros escolhidos e RMSE médio da busca (positivo).
print("Melhores parâmetros:", grid_search_ridge.best_params_)
print(f"RMSE médio da busca: {-grid_search_ridge.best_score_:.3f}")
grid_search_ridge.best_estimator_

In [ ]:
# 4. Candidatos ordenados pelo RMSE; rank 1 indica a melhor configuração.
df_busca_ridge = pd.DataFrame(grid_search_ridge.cv_results_)
df_busca_ridge["MAE"] = -df_busca_ridge["mean_test_neg_mean_absolute_error"]
df_busca_ridge["RMSE"] = -df_busca_ridge["mean_test_neg_root_mean_squared_error"]
df_busca_ridge.sort_values("rank_test_neg_root_mean_squared_error")[[
    "params", "mean_test_r2", "MAE", "RMSE",
    "std_test_neg_root_mean_squared_error",
]].head(10)

In [ ]:
# 5. Coeficientes do modelo reajustado, com os nomes das variáveis transformadas.
melhor_ridge = grid_search_ridge.best_estimator_
pipeline_ridge = melhor_ridge.regressor_
df_coeficientes_ridge = dataframe_coeficientes(
    pipeline_ridge["reg"].coef_,
    pipeline_ridge["preprocessor"].get_feature_names_out(),
)
plot_coeficientes(df_coeficientes_ridge, tituto="Coeficientes — Ridge")
df_coeficientes_ridge

## 5. Lasso

A penalização L1 pode zerar coeficientes. Aumentamos `max_iter` para dar tempo de convergir.

Seguimos os mesmos passos: definir a grade, treinar a busca, examinar os candidatos e visualizar os coeficientes.

In [ ]:
# 1. Grade exclusiva do Lasso; as opções comuns são iguais nos três modelos.
param_grid_lasso = {
    "regressor__reg__alpha": ALPHAS,
    "regressor__preprocessor": PREPROCESSADORES,
    "transformer": TRANSFORMADORES_ALVO,
}

# Mantemos o wrapper do alvo mesmo quando a grade escolhe transformer=None.
grid_search_lasso = grid_search_cv_regressor(
    regressor=Lasso(max_iter=100_000),
    param_grid=param_grid_lasso,
    preprocessor=preprocessamento_categoricas,
    target_transformer=target_transformer,
)

In [ ]:
# 2. Ajusta cada configuração por validação cruzada e reajusta a melhor em X, y.
grid_search_lasso.fit(X, y)

In [ ]:
# 3. Parâmetros escolhidos e RMSE médio da busca (positivo).
print("Melhores parâmetros:", grid_search_lasso.best_params_)
print(f"RMSE médio da busca: {-grid_search_lasso.best_score_:.3f}")
grid_search_lasso.best_estimator_

In [ ]:
# 4. Candidatos ordenados pelo RMSE; rank 1 indica a melhor configuração.
df_busca_lasso = pd.DataFrame(grid_search_lasso.cv_results_)
df_busca_lasso["MAE"] = -df_busca_lasso["mean_test_neg_mean_absolute_error"]
df_busca_lasso["RMSE"] = -df_busca_lasso["mean_test_neg_root_mean_squared_error"]
df_busca_lasso.sort_values("rank_test_neg_root_mean_squared_error")[[
    "params", "mean_test_r2", "MAE", "RMSE",
    "std_test_neg_root_mean_squared_error",
]].head(10)

In [ ]:
# 5. Coeficientes do modelo reajustado, com os nomes das variáveis transformadas.
melhor_lasso = grid_search_lasso.best_estimator_
pipeline_lasso = melhor_lasso.regressor_
df_coeficientes_lasso = dataframe_coeficientes(
    pipeline_lasso["reg"].coef_,
    pipeline_lasso["preprocessor"].get_feature_names_out(),
)
plot_coeficientes(df_coeficientes_lasso, tituto="Coeficientes — Lasso")
df_coeficientes_lasso

## 6. ElasticNet

Combina L1 e L2. Valores de `l1_ratio` próximos de 1 dão maior peso à penalização L1.

Seguimos os mesmos passos: definir a grade, treinar a busca, examinar os candidatos e visualizar os coeficientes.

In [ ]:
# 1. Grade exclusiva do ElasticNet; as opções comuns são iguais nos três modelos.
param_grid_elasticnet = {
    "regressor__reg__alpha": ALPHAS,
    "regressor__reg__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9],
    "regressor__preprocessor": PREPROCESSADORES,
    "transformer": TRANSFORMADORES_ALVO,
}

# Mantemos o wrapper do alvo mesmo quando a grade escolhe transformer=None.
grid_search_elasticnet = grid_search_cv_regressor(
    regressor=ElasticNet(max_iter=100_000),
    param_grid=param_grid_elasticnet,
    preprocessor=preprocessamento_categoricas,
    target_transformer=target_transformer,
)

In [ ]:
# 2. Ajusta cada configuração por validação cruzada e reajusta a melhor em X, y.
grid_search_elasticnet.fit(X, y)

In [ ]:
# 3. Parâmetros escolhidos e RMSE médio da busca (positivo).
print("Melhores parâmetros:", grid_search_elasticnet.best_params_)
print(f"RMSE médio da busca: {-grid_search_elasticnet.best_score_:.3f}")
grid_search_elasticnet.best_estimator_

In [ ]:
# 4. Candidatos ordenados pelo RMSE; rank 1 indica a melhor configuração.
df_busca_elasticnet = pd.DataFrame(grid_search_elasticnet.cv_results_)
df_busca_elasticnet["MAE"] = -df_busca_elasticnet["mean_test_neg_mean_absolute_error"]
df_busca_elasticnet["RMSE"] = -df_busca_elasticnet["mean_test_neg_root_mean_squared_error"]
df_busca_elasticnet.sort_values("rank_test_neg_root_mean_squared_error")[[
    "params", "mean_test_r2", "MAE", "RMSE",
    "std_test_neg_root_mean_squared_error",
]].head(10)

In [ ]:
# 5. Coeficientes do modelo reajustado, com os nomes das variáveis transformadas.
melhor_elasticnet = grid_search_elasticnet.best_estimator_
pipeline_elasticnet = melhor_elasticnet.regressor_
df_coeficientes_elasticnet = dataframe_coeficientes(
    pipeline_elasticnet["reg"].coef_,
    pipeline_elasticnet["preprocessor"].get_feature_names_out(),
)
plot_coeficientes(df_coeficientes_elasticnet, tituto="Coeficientes — ElasticNet")
df_coeficientes_elasticnet

## 7. Comparação dos modelos com validação cruzada aninhada

A **validação interna** escolhe os hiperparâmetros usando somente o treino de cada divisão externa. A **validação externa** mede o desempenho em dados que não participaram dessa escolha. Ridge, Lasso e ElasticNet usam as mesmas divisões e métricas.

Incluímos duas referências: prever sempre a média (`DummyRegressor`) e regressão linear com pré-processamento simples. Os modelos completos são avaliados, preservando a transformação do alvo escolhida pela busca. Essa etapa é mais demorada porque repete as buscas dentro de cada divisão externa.

Os gráficos de coeficientes anteriores descrevem os ajustes com todos os dados; não são os modelos de cada divisão externa.

In [ ]:
# clone remove os ajustes anteriores, preservando a configuração da busca.
# n_jobs=1 dentro de cada busca evita paralelismo em dois níveis.
modelos_comparacao = {
    "DummyRegressor": DummyRegressor(strategy="mean"),
    "LinearRegression": construir_pipeline_modelo_regressao(
        LinearRegression(), preprocessor=preprocessamento_simples,
    ),
    "Ridge": clone(grid_search_ridge).set_params(n_jobs=1),
    "Lasso": clone(grid_search_lasso).set_params(n_jobs=1),
    "ElasticNet": clone(grid_search_elasticnet).set_params(n_jobs=1),
}

resultados = {
    nome: cross_validate(
        modelo, X, y, cv=cv_externa, scoring=METRICAS,
        n_jobs=-1, error_score="raise",
    )
    for nome, modelo in modelos_comparacao.items()
}

df_resultados = organiza_resultados(resultados)
# Guardamos as colunas originais e acrescentamos os erros em escala positiva.
df_resultados["MAE"] = -df_resultados["test_neg_mean_absolute_error"]
df_resultados["RMSE"] = -df_resultados["test_neg_root_mean_squared_error"]
df_resultados

In [ ]:
# Médias e desvios entre as cinco divisões externas.
resumo_comparacao = df_resultados.groupby("model").agg(
    R2_medio=("test_r2", "mean"),
    R2_desvio=("test_r2", "std"),
    MAE_medio=("MAE", "mean"),
    RMSE_medio=("RMSE", "mean"),
    RMSE_desvio=("RMSE", "std"),
    tempo_medio_s=("time_seconds", "mean"),
).sort_values("RMSE_medio")
resumo_comparacao

In [ ]:
# A função de gráficos usa os nomes originais das métricas.
# Enviamos erros positivos para facilitar a leitura: menor MAE/RMSE é melhor.
df_graficos = df_resultados.copy()
df_graficos["test_neg_mean_absolute_error"] = df_graficos["MAE"]
df_graficos["test_neg_root_mean_squared_error"] = df_graficos["RMSE"]
plot_comparar_metricas_modelos(df_graficos)

### Como interpretar

- **R²:** quanto maior, melhor; pode ser negativo se o modelo for pior que prever a média.
- **MAE e RMSE:** quanto menores, melhor, em unidades do alvo original. O RMSE dá mais peso a erros grandes.
- **Desvio entre divisões:** mostra a variação do resultado; médias próximas não demonstram uma superioridade conclusiva.
- **Tempo:** para os regularizados, inclui a busca interna; as referências não fazem essa busca.
- **Coeficientes:** dependem do pré-processamento e da transformação do alvo escolhidos. Compare com cuidado suas magnitudes entre modelos.

Use o menor RMSE externo como critério principal, observando também a variação, o MAE e o custo computacional. `best_params_` de cada busca inicial descreve o ajuste final em todos os dados; cada divisão externa pode escolher parâmetros diferentes.